# CSW Metadata Crawler

This notebook crawls a CSW (OGC Catalog Service for the Web) endpoint and downloads
all metadata records as XML files into a configurable output directory.

## Prerequisites

Install dependencies (if not already done):

```bash
pip install requests lxml
```

## Shared output directory (Docker / local)

The `OUTPUT_DIR` path below is configurable. By default it points to
`../data/csw_metadata` (at the repository root); this directory is excluded from git.

To mount it inside a Docker service, add the following to `docker-compose.yml`:
```yaml
volumes:
  - ./data/csw_metadata:/data/csw_metadata
```

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — edit these values as needed
# ─────────────────────────────────────────────────────────────────────────────

# Base URL of the CSW service (without query parameters)
CSW_URL = "https://www.geocatalogue.fr/geonetwork/0e3541fb-db45-4166-982f-ee8363896023/fre/csw"

# Output directory for XML files.
# Accepts an absolute or relative path (relative to this notebook).
# This directory is excluded from git.
# Absolute example: "/home/user/data/csw_metadata"
OUTPUT_DIR = "../data/csw_metadata"

# Number of records per page (most CSW servers accept ≤ 500)
PAGE_SIZE = 100

# Delay between pages (seconds) — avoids overloading the server
REQUEST_DELAY = 0.5

In [ ]:
%pip install -r ../requirements-dev.txt

In [ ]:
import time
import requests
import xml.etree.ElementTree as ET
from pathlib import Path

# Register namespaces for clean serialisation
NAMESPACES = {
    "csw":   "http://www.opengis.net/cat/csw/2.0.2",
    "gmd":   "http://www.isotc211.org/2005/gmd",
    "gco":   "http://www.isotc211.org/2005/gco",
    "gml":   "http://www.opengis.net/gml",
    "srv":   "http://www.isotc211.org/2005/srv",
    "xlink": "http://www.w3.org/1999/xlink",
    "xsi":   "http://www.w3.org/2001/XMLSchema-instance",
    "ows":   "http://www.opengis.net/ows",
}

for prefix, uri in NAMESPACES.items():
    ET.register_namespace(prefix, uri)

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_path.resolve()}")

In [ ]:
# ── 1. Service check (GetCapabilities) ───────────────────────────────────────

resp = requests.get(
    CSW_URL,
    params={"SERVICE": "CSW", "VERSION": "2.0.2", "REQUEST": "GetCapabilities"},
    timeout=30,
)
resp.raise_for_status()

root_caps = ET.fromstring(resp.content)
service_id = root_caps.find(
    ".//{http://www.opengis.net/ows}Title"
    or ".//{http://www.opengis.net/cat/csw/2.0.2}Title"
)
title_text = service_id.text if service_id is not None else "(not found)"
print(f"CSW service available: {title_text}")
print(f"HTTP status: {resp.status_code}")

In [ ]:
# ── 2. Count total records ────────────────────────────────────────────────────

resp_hits = requests.get(
    CSW_URL,
    params={
        "SERVICE":    "CSW",
        "VERSION":    "2.0.2",
        "REQUEST":    "GetRecords",
        "RESULTTYPE": "hits",
        "TYPENAMES":  "gmd:MD_Metadata",
        "OUTPUTSCHEMA": "http://www.isotc211.org/2005/gmd",
        "NAMESPACE":  "xmlns(gmd=http://www.isotc211.org/2005/gmd)",
        "MAXRECORDS": 1,
    },
    timeout=30,
)
resp_hits.raise_for_status()

root_hits = ET.fromstring(resp_hits.content)
sr = root_hits.find(".//csw:SearchResults", NAMESPACES)
if sr is None:
    # fallback without namespace
    sr = root_hits.find(".//SearchResults")

total_records = int(sr.get("numberOfRecordsMatched", 0)) if sr is not None else 0
total_pages   = (total_records + PAGE_SIZE - 1) // PAGE_SIZE

print(f"Total records   : {total_records}")
print(f"Pages to fetch  : {total_pages} (page size: {PAGE_SIZE})")

In [ ]:
# ── 3. Paginated crawl — TopicCategory + Type sub-partitioning ───────────────
#
# GeoNetwork caps any GetRecords result set at 800, even with a filter.
# Strategy:
#   Pass A — each TopicCategory independently (with Type sub-partition if >= 800)
#   Pass B — unfiltered final pass (with Type sub-partition if >= 800)
# All passes share seen_ids for deduplication.
#
# Known server quirk: ELEMENTSETNAME=full causes some partitions to return 0
# records in results mode while hits mode reports > 0.  crawl_constraint()
# detects this and retries without ELEMENTSETNAME (server picks "summary").

ISO_TOPIC_CATEGORIES = [
    "farming", "biota", "boundaries", "climatologyMeteorologyAtmosphere",
    "economy", "elevation", "environment", "geoscientificInformation",
    "health", "imageryBaseMapsEarthCover", "intelligenceMilitary",
    "inlandWaters", "location", "oceans", "planningCadastre",
    "society", "structure", "transportation", "utilitiesCommunication",
]

ISO_TYPES = [
    "dataset", "series", "service", "software", "junk",
    "fieldSession", "collectionHardware", "collectionSession",
    "nonGeographicDataset", "dimensionGroup", "feature",
    "featureType", "propertyType", "attributeType", "attribute",
    "tile",
]

_CSW_NS  = "http://www.opengis.net/cat/csw/2.0.2"
_GMD_NS  = "http://www.isotc211.org/2005/gmd"
_GCO_NS  = "http://www.isotc211.org/2005/gco"
_TAG_SR  = f"{{{_CSW_NS}}}SearchResults"
_TAG_MD  = f"{{{_GMD_NS}}}MD_Metadata"
_TAG_FID = f"{{{_GMD_NS}}}fileIdentifier/{{{_GCO_NS}}}CharacterString"

CAP = 800


def _base_params(result_type: str, start: int, max_records: int,
                 constraint: str | None = None,
                 elementset: str | None = "full") -> dict:
    p = {
        "SERVICE":       "CSW",
        "VERSION":       "2.0.2",
        "REQUEST":       "GetRecords",
        "RESULTTYPE":    result_type,
        "TYPENAMES":     "gmd:MD_Metadata",
        "OUTPUTSCHEMA":  _GMD_NS,
        "NAMESPACE":     f"xmlns(gmd={_GMD_NS})",
        "STARTPOSITION": start,
        "MAXRECORDS":    max_records,
    }
    if elementset:
        p["ELEMENTSETNAME"] = elementset
    if constraint:
        p["CONSTRAINTLANGUAGE"]          = "CQL_TEXT"
        p["CONSTRAINT_LANGUAGE_VERSION"] = "1.1.0"
        p["CONSTRAINT"]                  = constraint
    return p


def count_records(csw_url: str, constraint: str | None = None) -> int:
    r = requests.get(csw_url,
                     params=_base_params("hits", 1, 1, constraint, elementset=None),
                     timeout=30)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    sr   = root.find(f".//{_TAG_SR}")
    if sr is None:
        return 0
    return int(sr.get("numberOfRecordsMatched", 0))


def fetch_page(csw_url: str, start: int, max_records: int,
               constraint: str | None = None,
               elementset: str | None = "full") -> bytes:
    r = requests.get(csw_url,
                     params=_base_params("results", start, max_records,
                                         constraint, elementset),
                     timeout=60)
    r.raise_for_status()
    return r.content


def extract_and_save(xml_bytes: bytes, output_dir: Path, seen: set) -> tuple[int, int]:
    root = ET.fromstring(xml_bytes)
    saved = dupes = 0
    for i, rec in enumerate(root.findall(f".//{_TAG_MD}")):
        file_id_el = rec.find(_TAG_FID)
        file_id    = (file_id_el.text.strip()
                      if file_id_el is not None and file_id_el.text else None)
        safe_name  = (
            file_id.replace("/","_").replace("\\","_").replace(":","_").replace(" ","_")
            if file_id else f"record_{len(seen)+i:05d}_unknown_id"
        )
        if safe_name in seen:
            dupes += 1
            continue
        seen.add(safe_name)
        (output_dir / f"{safe_name}.xml").write_text(
            '<?xml version="1.0" encoding="UTF-8"?>\n' + ET.tostring(rec, encoding="unicode"),
            encoding="utf-8",
        )
        saved += 1
    return saved, dupes


def crawl_constraint(csw_url: str, output_path: Path, seen_ids: set,
                     constraint: str | None, label: str,
                     elementset: str | None = "full") -> int:
    """Paginate through one constraint. If ELEMENTSETNAME=full causes the server
    to return 0 records on the first page, retry without it automatically.
    """
    total_saved = 0
    start = 1
    retried_elementset = False

    while start > 0:
        try:
            xml_bytes = fetch_page(csw_url, start, PAGE_SIZE, constraint, elementset)
            saved, dupes = extract_and_save(xml_bytes, output_path, seen_ids)

            root_page = ET.fromstring(xml_bytes)
            sr        = root_page.find(f".//{_TAG_SR}")
            n         = int(sr.get("numberOfRecordsMatched", 0)) if sr is not None else 0
            next_rec  = int(sr.get("nextRecord", 0))             if sr is not None else 0

            # Server returned 0 matched on first page despite count > 0:
            # retry without ELEMENTSETNAME (server-side bug with full + some constraints)
            if start == 1 and n == 0 and not retried_elementset and elementset == "full":
                print(f"    [retrying without ELEMENTSETNAME=full…]", flush=True)
                retried_elementset = True
                elementset = None
                continue

            total_saved += saved
            print(f"    {start}…{min(start+PAGE_SIZE-1,n) if n else '?'}/{n}"
                  f"  +{saved} ({dupes} dupes)", flush=True)

            if next_rec == 0 or next_rec <= start:
                break
            start = next_rec

        except requests.HTTPError as exc:
            print(f"    HTTP error at {start}: {exc}")
            start += PAGE_SIZE
        except ET.ParseError as exc:
            print(f"    XML parse error at {start}: {exc}")
            start += PAGE_SIZE
        time.sleep(REQUEST_DELAY)
    return total_saved


def crawl_partition(csw_url: str, output_path: Path, seen_ids: set,
                    constraint: str | None = None, label: str = "") -> int:
    """Count, then crawl. Sub-partition by Type if count >= CAP."""
    n = count_records(csw_url, constraint)
    print(f"  [{label}]  {n} records", flush=True)
    if n == 0:
        return 0

    if n < CAP:
        return crawl_constraint(csw_url, output_path, seen_ids, constraint, label)

    print(f"    → {n} >= {CAP}, sub-partitioning by Type…", flush=True)
    total_saved = 0
    for iso_type in ISO_TYPES:
        if constraint and "TopicCategory" in constraint:
            sub_c = f"TopicCategory='{label}' AND Type='{iso_type}'"
        elif not constraint:
            sub_c = f"Type='{iso_type}'"
        else:
            sub_c = f"({constraint}) AND Type='{iso_type}'"

        sub_n = count_records(csw_url, sub_c)
        print(f"    [{label}/{iso_type}]  {sub_n} records", flush=True)
        if sub_n == 0:
            continue
        total_saved += crawl_constraint(csw_url, output_path, seen_ids,
                                        sub_c, f"{label}/{iso_type}")
        time.sleep(REQUEST_DELAY)
    return total_saved


# ─── Main crawl ───────────────────────────────────────────────────────────────
seen_ids    = {f.stem for f in output_path.glob("*.xml")}
total_saved = 0

print(f"Already on disk : {len(seen_ids)} records.")

print(f"\n── Pass A : TopicCategory (+ Type sub-partition if needed) ──")
for category in ISO_TOPIC_CATEGORIES:
    total_saved += crawl_partition(
        CSW_URL, output_path, seen_ids,
        constraint=f"TopicCategory='{category}'",
        label=category,
    )

print(f"\n── Pass B : unfiltered ───────────────────────────────────────")
total_saved += crawl_partition(CSW_URL, output_path, seen_ids, label="unfiltered")

on_disk = len(list(output_path.glob("*.xml")))
print(f"\n{'─'*55}")
print(f"  New records saved : {total_saved}")
print(f"  Total on disk     : {on_disk}  /  {total_records}")
print(f"  Output dir        : {output_path.resolve()}")


In [ ]:
# ── 4. Quick verification of downloaded content ───────────────────────────────

xml_files = sorted(output_path.glob("*.xml"))
print(f"XML files present: {len(xml_files)}")

# Preview of the first 5 files
for f in xml_files[:5]:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name}  ({size_kb:.1f} KB)")